# Diffusion inside a sphere: manual test, part 2
The tutorial-data fit, the fit loop (minimizers, a fixed parameter, sampling) and the EISF fit, in a fresh kernel: dependent-Parameter updates slow down as the global object map grows (bug 12 in part 1), so the 6-Q models of part 1 would make these fits several times slower.

In [1]:
import numpy as np
import scipp as sc
from scipp.constants import hbar

import easydynamics as edyn

In [2]:
Q = sc.linspace('Q', 0.4, 2.0, 6, unit='1/angstrom')
energy = sc.linspace('energy', -2.0, 2.0, 401, unit='meV')
Q_fine = sc.linspace('Q', 0.02, 3.0, 300, unit='1/angstrom')
hbar_meV_ps = hbar.to(unit='meV*ps').value


def versus_Q(values, unit='dimensionless'):
    """A model quantity on Q_fine as a DataArray, for plopp."""
    return sc.DataArray(sc.array(dims=['Q'], values=values, unit=unit), coords={'Q': Q_fine})

## Basic fit to the tutorial data
`fake_advanced_data.hdf5` from the tutorials, rebinned to 2 Q values. That data was made from a
Gaussian, a Lorentzian, a damped harmonic oscillator and a sloped background, not from this model,
so this only checks that loading and fitting run on a real file; there is no background or
resolution in this model.

In [3]:
tutorial = edyn.Experiment(display_name='Tutorial')
tutorial.load_hdf5(filename='../../docs/docs/tutorials/data/fake_advanced_data.hdf5')
tutorial.rebin({'Q': 2, 'energy': 256})
tutorial_model = edyn.DiffusionInSphere(scale=1.0, radius=2.0, diffusion_coefficient=0.1)
tutorial_analysis = edyn.Analysis(
    experiment=tutorial, sample_model=edyn.SampleModel(diffusion_models=tutorial_model)
)
tutorial_analysis.fix_energy_offset()
tutorial_results = tutorial_analysis.fit(fit_method='simultaneous')
{'success': all(result.success for result in tutorial_results)} | {
    parameter.name: f'{parameter.value:.4g} ± {parameter.error:.2g} {parameter.unit}'
    for parameter in tutorial_model.get_global_variables()
}

/home/mark/Projects/dynamics-lib/src/easydynamics/sample_model/diffusion_model/delta_lorentzian_diffusion_model_base.py:509: UserWarning: Truncating the sum at number_of_lorentzians=10 omits more than 0.001 of the spectral weight at some Q values. At most 98 terms are available (the roots below 26.84, the same limit as Mantid); with all of them this happens from QR of about 10.
  self.create_component_collections()
CollectionBase is deprecated and will be removed in a future version. Please migrate to ModelBase or EasyList.


{'success': True,
 'scale': '21.89 ± 1.7 meV',
 'radius': '5.695 ± 0.13 Å',
 'diffusion_coefficient': '1.817e-08 ± 2.3e-09 m^2/s'}

## Fit loop: minimizers, fixed parameters, sampling and timings
One Q value (10 Lorentzians, the default), so that the dependent Parameters keep it quick. Every
minimizer must recover $\tau$ and $\sigma$ (within 5 standard errors where it reports errors,
else within 5 %); the fixed radius must not move; a short Bumps sampling run must work and label
its columns with the parameters. The times are measured, not asserted: the component areas and
widths are dependent Parameters, whose updates are slow in EasyScience Core 2.5.1.

In [4]:
import time

from easyscience.fitting import AvailableMinimizers

loop_Q = sc.array(dims=['Q'], values=[1.0], unit='1/angstrom')
loop_shape = (1, energy.shape[0])


def loop_data(values, variances):
    return sc.DataArray(
        sc.array(dims=['Q', 'energy'], values=values, variances=variances),
        coords={'Q': loop_Q, 'energy': energy},
    )


loop_truth = {'radius': 3.0, 'diffusion_coefficient': 0.1}  # D in angstrom^2/ps
loop_experiment = edyn.Experiment('loop', data=loop_data(np.ones(loop_shape), np.ones(loop_shape)))
start = time.perf_counter()
loop_true_model = edyn.DiffusionInSphere(**loop_truth)
loop_true_analysis = edyn.Analysis(
    experiment=loop_experiment, sample_model=edyn.SampleModel(diffusion_models=loop_true_model)
)
loop_clean = np.array(loop_true_analysis.calculate())
build_seconds = time.perf_counter() - start
start = time.perf_counter()
loop_true_model.radius = 3.1
change_seconds = time.perf_counter() - start
loop_true_model.radius = 3.0
loop_sigma = 0.003 * loop_clean.max()
loop_experiment.data = loop_data(
    loop_clean + np.random.default_rng(seed=2).normal(0.0, loop_sigma, loop_shape),
    np.full(loop_shape, loop_sigma**2),
)


def loop_fit(minimizer):
    loop_model = edyn.DiffusionInSphere(radius=2.5, diffusion_coefficient=0.07)
    # D is stored in m^2/s (about 1e-9), where DFO stalls after its first step (bug 13 in part 1);
    # nm^2/ns (shown by Scipp as nGy*s, the same dimension) gives it a value of order 1
    loop_model.diffusion_coefficient.convert_unit('nm**2/ns')
    analysis = edyn.Analysis1d(
        experiment=loop_experiment,
        sample_model=edyn.SampleModel(diffusion_models=loop_model),
        Q_index=0,
    )
    analysis.fix_energy_offset()
    analysis.fitter.switch_minimizer(minimizer)
    start = time.perf_counter()
    analysis.fit()
    seconds = time.perf_counter() - start
    for name, true_value in loop_truth.items():
        parameter = getattr(loop_model, name)
        unit = 'angstrom' if name == 'radius' else 'angstrom**2/ps'
        true_value = sc.scalar(true_value, unit=unit).to(unit=parameter.unit).value
        if parameter.error:
            assert abs(parameter.value - true_value) < 5 * parameter.error, (minimizer, name)
        else:
            assert abs(parameter.value - true_value) < 0.05 * true_value, (minimizer, name)
    return loop_model, analysis, seconds


loop_results = {}
for minimizer in (AvailableMinimizers.LMFit, AvailableMinimizers.Bumps, AvailableMinimizers.DFO):
    loop_model, loop_analysis, seconds = loop_fit(minimizer)
    loop_results[minimizer.name] = {
        'fit (s)': round(seconds, 2),
        **{
            name: f'{getattr(loop_model, name).value:.4g} ± {getattr(loop_model, name).error:.2g}'
            for name in loop_truth
        },
    }
{
    'build, 1 Q (s)': round(build_seconds, 2),
    'one change of the radius (s)': round(change_seconds, 3),
} | loop_results

{'build, 1 Q (s)': 0.1,
 'one change of the radius (s)': 0.038,
 'LMFit': {'fit (s)': 0.94,
  'radius': '2.998 ± 0.004',
  'diffusion_coefficient': '1.006 ± 0.0097'},
 'Bumps': {'fit (s)': 14.62,
  'radius': '2.998 ± 0.0039',
  'diffusion_coefficient': '1.006 ± 0.0096'},
 'DFO': {'fit (s)': 3.46,
  'radius': '2.998 ± 0.00088',
  'diffusion_coefficient': '1.006 ± 0.0025'}}

In [5]:
# A fixed parameter must stay where it is during a fit
fixed_model = edyn.DiffusionInSphere(radius=2.5, diffusion_coefficient=0.07)
fixed_model.radius.fixed = True
fixed_analysis = edyn.Analysis1d(
    experiment=loop_experiment,
    sample_model=edyn.SampleModel(diffusion_models=fixed_model),
    Q_index=0,
)
fixed_analysis.fix_energy_offset()
fixed_analysis.fit()
# Exact on purpose: a fixed parameter must not move at all.
assert fixed_model.radius.value == 2.5, 'the fixed radius moved'  # ruff: ignore[float-equality-comparison]
{'radius (fixed)': fixed_model.radius.value, 'D': str(fixed_model.diffusion_coefficient)}

{'radius (fixed)': np.float64(2.5),
 'D': "<Parameter 'diffusion_coefficient': 5.862e-10 ± 4.453e-11 m^2/s, bounds=[1.0000000000000001e-20:inf]>"}

In [6]:
loop_model, loop_analysis, _ = loop_fit(AvailableMinimizers.LMFit)
# The bounds are the prior of the sampler, so every free parameter needs finite ones
loop_analysis.bayesian.suggest_bounds().apply()
start = time.perf_counter()
sampling = loop_analysis.bayesian.sample(samples=200, burn=20, thin=2)
sampling_seconds = time.perf_counter() - start
# Core's chain columns are unique names (Parameter_123); EasyDynamics labels them by parameter name
labels = [entry.name for entry in loop_analysis.bayesian.summary().entries]
assert any('radius' in label for label in labels), labels
assert any('diffusion_coefficient' in label for label in labels), labels
{
    'sampling (s)': round(sampling_seconds, 1),
    'draws': sampling.draws.shape,
    'chain columns': sampling.param_names,
    'labels': labels,
}

/home/mark/Projects/dynamics-lib/.pixi/envs/default/lib/python3.15/site-packages/bumps/dream/convergence.py:186: UserWarning: Did not converge!
  warnings.warn("Did not converge!")


{'sampling (s)': 72.0,
 'draws': (87, 3),
 'chain columns': ['Parameter_11112', 'Parameter_11113', 'Parameter_11114'],
 'labels': ['scale', 'radius', 'diffusion_coefficient']}

## EISF against Q with ParameterAnalysis
The equivalent of Mantid's `EISFDiffSphere`: synthetic elastic areas against Q, fitted with the
`elastic_area` target of the model. The EISF does not depend on D, so D is fixed.

In [7]:
Q_eisf = sc.linspace('Q', 0.2, 2.0, 15, unit='1/angstrom')
eisf_noise = 0.01
true_eisf = edyn.DiffusionInSphere(radius=3.0).calculate_EISF(Q_eisf.values)
eisf_data = sc.Dataset({
    'Delta function area': sc.DataArray(
        sc.array(
            dims=['Q'],
            values=true_eisf + np.random.default_rng(seed=3).normal(0.0, eisf_noise, 15),
            variances=np.full(15, eisf_noise**2),
            unit='meV',
        ),
        coords={'Q': Q_eisf},
    )
})
eisf_model = edyn.DiffusionInSphere(radius=2.5)
eisf_model.diffusion_coefficient.fixed = True
binding = edyn.FitBinding(model=eisf_model, targets={'elastic_area': 'Delta function area'})
eisf_analysis = edyn.ParameterAnalysis(parameters=eisf_data, bindings=binding)
eisf_analysis.fit()
assert abs(eisf_model.radius.value - 3.0) < 5 * eisf_model.radius.error
{'radius': f'{eisf_model.radius.value:.4f} ± {eisf_model.radius.error:.2g} Å'}

CollectionBase is deprecated and will be removed in a future version. Please migrate to ModelBase or EasyList.


{'radius': '3.0372 ± 0.033 Å'}